# 📈 MobiMarket 시계열 데이터 분석 및 최적 매수 타이밍 도출

이 노트북은 마비노기 모바일 거래소 수집 데이터(`mobi_market.db`)를 바탕으로 다음을 분석합니다:
1. **기술적 지표 계산**: 5일/20일 이동평균선(SMA), 볼린저 밴드(Bollinger Bands, 20일 2σ)
2. **시간대별/요일별 가격 패턴**: 어느 시간대에 매수하는 것이 가장 저렴한지
3. **매수/매도 포지션 시그널 도출**: 저점 덤핑 매물 포착 및 차익 실현 구간 제시

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 한글 폰트 설정 (Windows Malgun Gothic)
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="darkgrid")

print("환경 초기화 완료")

In [ ]:
# SQLite DB 로드
db_candidates = [
    Path("../target/release/mobi_market.db"),
    Path("../mobi_market.db"),
    Path("target/release/mobi_market.db")
]
db_path = next((p for p in db_candidates if p.exists()), None)
assert db_path is not None, "mobi_market.db 파일을 찾을 수 없습니다."

conn = sqlite3.connect(str(db_path))
conn.execute("PRAGMA busy_timeout = 5000;")
df = pd.read_sql_query("SELECT item_name, min_unit_price, in_stock, scraped_at FROM market_snapshots ORDER BY id ASC", conn)
conn.close()

df['scraped_at'] = pd.to_datetime(df['scraped_at'])
df['hour'] = df['scraped_at'].dt.hour
df['day_name'] = df['scraped_at'].dt.day_name()
print(f"총 {len(df):,}개 레코드 로드 완료. 품목 수: {df['item_name'].nunique()}개")
df.head()

In [ ]:
# 주요 품목 이동평균 및 볼린저 밴드 시각화
target_item = "마력 깃든 용비늘"
sub = df[df['item_name'] == target_item].copy()
sub.sort_values('scraped_at', inplace=True)

sub['SMA_5'] = sub['min_unit_price'].rolling(5, min_periods=1).mean()
sub['SMA_20'] = sub['min_unit_price'].rolling(20, min_periods=1).mean()
sub['STD_20'] = sub['min_unit_price'].rolling(20, min_periods=1).std().fillna(0)
sub['BB_Upper'] = sub['SMA_20'] + (sub['STD_20'] * 2)
sub['BB_Lower'] = sub['SMA_20'] - (sub['STD_20'] * 2)

plt.figure(figsize=(12, 6))
plt.plot(sub['scraped_at'], sub['min_unit_price'], label='최저가 실측치', color='#E5A93C', marker='o', linewidth=2)
plt.plot(sub['scraped_at'], sub['SMA_20'], label='20회 이동평균(SMA)', color='#58A6FF', linestyle='--')
plt.fill_between(sub['scraped_at'], sub['BB_Lower'], sub['BB_Upper'], color='#58A6FF', alpha=0.15, label='볼린저 밴드 (±2σ)')
plt.title(f"{target_item} 시세 추이 및 볼린저 밴드", fontsize=14, fontweight='bold')
plt.ylabel("단가 (데카)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 시간대별 최저가 분포 분석
hourly = sub.groupby('hour')['min_unit_price'].mean().reset_index()

plt.figure(figsize=(10, 4))
sns.barplot(data=hourly, x='hour', y='min_unit_price', palette='magma')
plt.title(f"{target_item} 시간대별 평균 시세 (골든 타임 분석)", fontsize=13)
plt.xlabel("시간대 (24시 기준)")
plt.ylabel("평균 최저가 (데카)")
plt.tight_layout()
plt.show()